# 07 — Anomaly Detection

**Wind Turbine Predictive Maintenance & Failure Intelligence System**

## Why a different approach
NB06 (supervised) learned "what pre-fault looks like" from only 12 events — hard,
and blind to generator-bearing faults. Anomaly detection inverts the problem:
learn **what normal operation looks like** from abundant normal data, then flag
deviations. It needs no fault examples, so it may catch faults supervised missed.

## Method & leakage discipline
- Train Isolation Forest & LOF on **normal operating data only**.
- Evaluate leave-turbines-out: fit on other turbines' normal data, score the
  held-out turbine, check whether anomaly scores rise in its pre-fault windows.
- Compare against NB06's supervised result — especially on generator-bearing faults.

## The question
Does "deviation from normal" flag faults that supervised classification couldn't?

In [1]:
import os
from pathlib import Path
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 120)

BASE = Path("..") / "data" / "raw" / "Wind Farm A"
FEATURES_DIR = Path("..") / "data" / "processed" / "features"

events = pd.read_csv(BASE / "event_info.csv", sep=";")
feature_list = pd.read_csv(FEATURES_DIR / "feature_list.csv")["feature"].tolist()

fm = pd.read_csv(FEATURES_DIR / "feature_matrix.csv")
labels = pd.read_csv(FEATURES_DIR / "labels_and_folds.csv")
data = fm.merge(labels[["id", "event_id", "target", "fold"]], on=["id", "event_id"], how="inner")

fault_map = events.set_index("event_id")["event_description"].to_dict()
data["fault_type"] = data["event_id"].map(fault_map).fillna("normal")

print("Data:", data.shape, "| target rate:", round(data["target"].mean(), 4))
print("Rows available for 'normal' training (target==0):", (data["target"] == 0).sum())

Data: (1195779, 190) | target rate: 0.0167
Rows available for 'normal' training (target==0): 1175811


## 1. Isolation Forest — leave-turbines-out

For each held-out turbine, we fit Isolation Forest on the **normal-operation rows
of the other turbines**, then score every row of the held-out turbine. Higher
anomaly score = more unlike normal. We then check whether scores rise in the
held-out turbine's pre-fault windows — the same honest, unseen-turbine test as NB06.

Isolation Forest is used first: it scales to large data and needs no scaling. LOF
(local, slower) follows on a subsample.

In [2]:
from sklearn.ensemble import IsolationForest
from sklearn.metrics import average_precision_score

# Subsample normal training rows per fit for speed (IsoForest doesn't need all 1.2M)
RNG = np.random.default_rng(42)

data["iso_score"] = np.nan

for f in sorted(data["fold"].unique()):
    test_mask = data["fold"] == f
    # TRAIN on other turbines' NORMAL rows only (no fault leakage, no held-out turbine)
    train_mask = (~test_mask) & (data["target"] == 0)
    train_idx = np.where(train_mask.values)[0]
    # subsample to 150k for speed
    if len(train_idx) > 150000:
        train_idx = RNG.choice(train_idx, 150000, replace=False)

    Xtr = data.iloc[train_idx][feature_list].values
    Xte = data.loc[test_mask, feature_list].values

    iso = IsolationForest(n_estimators=200, contamination="auto",
                          max_samples=min(len(train_idx), 50000),
                          n_jobs=-1, random_state=42)
    iso.fit(Xtr)
    # score_samples: higher = more normal; negate so higher = more anomalous
    data.loc[test_mask, "iso_score"] = -iso.score_samples(Xte)

    test_turbine = data.loc[test_mask, "asset_id"].iloc[0]
    ap = average_precision_score(data.loc[test_mask, "target"],
                                 data.loc[test_mask, "iso_score"])
    print(f"  fold {f} (turbine {test_turbine}): anomaly PR-AUC = {ap:.4f}")

overall_ap = average_precision_score(data["target"], data["iso_score"])
print(f"\nIsolation Forest overall PR-AUC: {overall_ap:.4f}  (baseline {data['target'].mean():.4f})")

  fold 0 (turbine 10): anomaly PR-AUC = 0.0509
  fold 1 (turbine 0): anomaly PR-AUC = 0.0259
  fold 2 (turbine 11): anomaly PR-AUC = 0.0102
  fold 3 (turbine 13): anomaly PR-AUC = 0.0126
  fold 4 (turbine 21): anomaly PR-AUC = 0.0164

Isolation Forest overall PR-AUC: 0.0261  (baseline 0.0167)


In [3]:
# Event-level: does the anomaly score rise in each fault's pre-fault window?
# And crucially — does it catch the events supervised MISSED?
oof = pd.read_csv(Path("..") / "reports" / "model_results" / "oof_predictions.csv")
data = data.merge(oof[["id", "event_id", "oof_pred"]], on=["id", "event_id"], how="left")

rows = []
for eid in data.loc[data["target"] == 1, "event_id"].unique():
    ev = data[data["event_id"] == eid]
    inside = ev["target"] == 1
    rows.append({
        "event_id": eid,
        "fault_type": ev["fault_type"].iloc[0],
        "iso_ratio": round(ev.loc[inside,"iso_score"].mean() / (ev.loc[~inside,"iso_score"].mean()+1e-9), 2),
        "sup_ratio": round(ev.loc[inside,"oof_pred"].mean() / (ev.loc[~inside,"oof_pred"].mean()+1e-9), 2),
    })

cmp = pd.DataFrame(rows)
cmp["iso_detects"] = cmp["iso_ratio"] > 1
cmp["sup_detects"] = cmp["sup_ratio"] > 1
print("Event-level: anomaly (iso) vs supervised (sup) risk ratio inside/outside fault window:")
print(cmp.sort_values("iso_ratio", ascending=False).to_string(index=False))
print(f"\nIsoForest detects (ratio>1): {cmp['iso_detects'].sum()}/12")
print(f"Supervised detects (ratio>1): {cmp['sup_detects'].sum()}/12")
print(f"Caught by EITHER method: {(cmp['iso_detects'] | cmp['sup_detects']).sum()}/12")

Event-level: anomaly (iso) vs supervised (sup) risk ratio inside/outside fault window:
 event_id                fault_type  iso_ratio  sup_ratio  iso_detects  sup_detects
       40 Generator bearing failure       1.06       0.02         True        False
        0 Generator bearing failure       1.04       1.56         True         True
       73           Hydraulic group       1.03       1.74         True         True
       45           Hydraulic group       1.02       0.09         True        False
       22           Hydraulic group       1.01       3.83         True         True
       42           Hydraulic group       1.01       0.89         True        False
       68       Transformer failure       1.00       5.87        False         True
       10           Gearbox failure       0.99       2.91        False         True
       26           Hydraulic group       0.98       4.96        False         True
       51  Gearbox bearings damaged       0.98      10.42        False   

### The key result — supervised and anomaly detection are complementary

Event-level detection (risk ratio inside vs outside each fault window, held-out turbines):

| Method | Faults detected (ratio > 1) |
|---|---|
| Supervised (LightGBM) | 9 / 12 |
| Anomaly detection (Isolation Forest) | 6 / 12 |
| **Either method (combined)** | **12 / 12** |

- **They catch different faults.** Supervised excels at faults with consistent
  learnable signatures — gearbox/mechanical (ratios up to 12×). Anomaly detection
  flags faults that deviate from normal but lack a consistent supervised pattern —
  notably the **generator-bearing failure (event 40)** that supervised missed
  entirely (0.02), and hydraulic event 45.
- **Combined, they detect all 12 faults** on turbines never seen in training —
  neither method reaches this alone.
- **Honest caveat:** anomaly ratios are modest in magnitude (~1.0–1.06); IsoForest's
  absolute signal is weak. Its value is *coverage of supervised's blind spots*, not
  raw strength. This motivates fusing both signals into one risk score (NB10).

**This complementarity — different fault mechanisms need different detection
strategies — is the project's central finding.**